# PySpark vs SQL!📒

🧠 **Regla mental para repetir siempre:**  
        * **SQL:** Declarativo (le dices al motor *qué* quieres obtener y él optimiza el cómo).  
        * **PySpark:** Transformaciones encadenadas sobre DataFrames (defines los pasos y operaciones de forma secuencial).  


In [0]:
%sql
select * from samples.bakehouse.sales_customers
limit 10

In [0]:
# Declaro la tabla que voy a usar: df = spark.read.table('catalogo.schema.tabla')
# Hablando en terminos de SQL este seria mi From:

df = spark.read.table('samples.bakehouse.sales_customers')

# Ahora voy a realizar el Select:

df_selected = df.select('*')

df_selected.display()

#DISTINCT🏙️

In [0]:
%sql
select distinct country
from samples.bakehouse.sales_customers

In [0]:
df_distinct = df_selected.select('country').distinct()

df_distinct.display()

#WHERE/FILTRO 🧩

In [0]:
%sql
SELECT * FROM samples.bakehouse.sales_customers
WHERE country == 'Japan' and gender == 'female'
limit 5 ;

In [0]:
# importo la libreria de funciones para filtrar por columna (col) en Pyspark:
from pyspark.sql.functions import col

# Filtro por el pais de USA:
df_filtered = df_selected.filter(
    (col('country') == 'USA') &
    (col('gender') == 'female')
     )

df_filtered.limit(4).display()

#ORDER BY + LIMIT 🧩

In [0]:
%sql
select * from samples.bakehouse.sales_customers
order by first_name asc
limit 10;


In [0]:
df_sorted = df_filtered.orderBy(col('first_name').asc())

df_sorted.limit(4).display()

# CREATE COLUMN 📊

In [0]:
%sql
select 
  concat(first_name, ' ', last_name) as full_name,
  concat_ws(', ', city, state, country, continent) as address, -- concat_ws me indica que separa con coma y espacio (, ) entre palabras.
  35 as age,
  "random coment" as comment,
  true as is_true
from samples.bakehouse.sales_customers
limit 10;

In [0]:
from pyspark.sql.functions import col, concat, lit # lit es para declarar una constante que se repetira.
df_plus_columns = df.select(
    concat(col('first_name'), lit(' '), col('last_name')).alias('full_name'),
    lit(True).alias('is_true'),
    lit('random coment').alias('comment')
).limit (10)
display(df_plus_columns)

# GROUP BY + AGGREGATION

In [0]:
%sql
select country, count(1) AS cantidad
from samples.bakehouse.sales_franchises
group by country
order by cantidad desc

In [0]:
from pyspark.sql.functions import count, col

df_grouped = (
    df.groupBy('country')
    .agg(count('*').alias('count'))
    .orderBy(col('count').desc())
)

df_grouped.display()

# JOIN (CLAVE PARA SILVER Y GOLD)🖇️

In [0]:
%sql
SELECT 
    concat(c.first_name, " ", c.last_name) AS full_name,
    c.country,
    c.gender,
    s.product,
    s.quantity,
    s.paymentMethod,
    s.dateTime
FROM samples.bakehouse.sales_transactions s
JOIN samples.bakehouse.sales_customers c
    ON s.customerID = c.customerID;

In [0]:
df_sales = spark.read.table('samples.bakehouse.sales_transactions')

df_joined = (
    df.join(df_sales, on='customerID', how='inner')
    .select(('*'))
)
df_joined.display()

# CAST

In [0]:
%sql
SELECT 
    CAST(transactionID AS STRING) AS transactionID,
    CAST(customerID AS STRING) AS customerID,
    CAST(franchiseID AS STRING) AS franchiseID,
    CAST(dateTime AS STRING) AS dateTime,
    CAST(product AS STRING) AS product
FROM samples.bakehouse.sales_transactions;

In [0]:
df_casted = df_sales.select(
    col("transactionID").cast("bigint"),
    col("customerID").cast("string"),
    col("franchiseID").cast("string"),
    col("dateTime").cast("string"),
    col("product"),
    col("quantity").cast("bigint"),
    col("unitPrice").cast("double"),
    
    (col("unitPrice") * col("quantity")).cast("bigint").alias("totalPrice"),
)

df_casted.display()

# CREACION DE COLUMNAS CON WITH COLUMN

In [0]:
df_test = df_casted.withColumn("nueva_columnita", lit("nueva_columnita_literal!!!!!"))
df_test.display()

# ELIMINAR COLUMNA CON DROP☠️

In [0]:
df_test = df_casted.withColumn("nueva_columnita", lit("nueva_columnita_literal!!!!!"))

df_new = df_test.drop("nueva_columnita")
df_new.display()